# Imports

In [ ]:
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from uavloc.ardupilot.mission import MissionPlan, mission_items, write_waypoints
from uavloc.ardupilot.params import vehicle_params, write_param_file
from uavloc.ardupilot.scripts import lua_script_path
from uavloc.builds import build_dir
from uavloc.data.satellite import GeoImage
from uavloc.data.tiles import ESRI_WORLD_IMAGERY, geo_mosaic
from uavloc.features.onnx_segmenter import OnnxSegmenter
from uavloc.geo import LatLon
from uavloc.onboard.corridor import corridor_width
from uavloc.onboard.package import load_package, newest_package
from uavloc.onboard.package_writer import PackageRequest, write_package
from uavloc.onboard.replay import replay_stub
from uavloc.planning.config import MissionInputs, PlannerConfig
from uavloc.planning.localizability import SelfSimilarity
from uavloc.planning.planner import plan
from uavloc.plotting.planning import show_raster
from uavloc.plotting.segmentation import colour_labels
from uavloc.simulation.camera import CameraConfig
from uavloc.track import REPO_ROOT

# Params

In [ ]:
SLUG = "onboard-runner"
START = (49.905, 23.885)        # west of Birky
END = (49.915, 23.925)          # east of Birky
SATELLITE_ONNX = "pocs/lraspp-pi5/models/lraspp-satellite.onnx"
DRONE_ONNX = "pocs/lraspp-pi5/models/lraspp-drone.onnx"
TILE_CACHE = "data/derived/tile-cache"
PLANNER_CACHE = "data/derived/planner-cache"
MISSION_ID = "birky-verify"
REPLAY_PERIOD_S = 0.5
STUB_NOISE_M = 6.0
STUB_SEED = 0
ZOOM_HALF_SIZE_M = 150.0        # half-width of the window around the longest blind stretch

# Data

In [ ]:
build = build_dir(SLUG, create=True)
figures = build / "figures"
package_dir = build / "runs" / "packages"
shutil.rmtree(package_dir, ignore_errors=True)
inputs = MissionInputs(start=LatLon(*START), end=LatLon(*END))
config = PlannerConfig()

# Model

In [ ]:
segmenter = OnnxSegmenter(REPO_ROOT / SATELLITE_ONNX)
localizability = SelfSimilarity(config.matcher, config.match_gsd_m, config.blind_s_max_m, config.blind_c_min)

# Run

In [ ]:
result = plan(inputs, config, ESRI_WORLD_IMAGERY, segmenter, localizability,
              REPO_ROOT / TILE_CACHE, REPO_ROOT / PLANNER_CACHE)
print(f"{len(result.routes)} routes; blind limit {result.blind_limit_m:.0f} m")

In [ ]:
def lowest_blind_route(routes):
    return min(routes, key=lambda route: (route.blind_share, route.length_m))

In [ ]:
route = lowest_blind_route(result.routes)
print(f"w={route.weight:g}: {route.length_m:.0f} m, blind {route.blind_share:.1%}, longest blind {route.longest_blind_m:.0f} m")

In [ ]:
satellite = geo_mosaic(ESRI_WORLD_IMAGERY, result.area, config.zoom, REPO_ROOT / TILE_CACHE)

In [ ]:
def write_ardupilot_files(route, inputs, directory) -> dict:
    directory.mkdir(parents=True, exist_ok=True)
    waypoints, params = directory / "mission.waypoints", directory / "uavloc.param"
    write_waypoints(waypoints, mission_items(MissionPlan.from_route(route, inputs)))
    write_param_file(params, vehicle_params())
    return {"mission.waypoints": waypoints, "uavloc.param": params, "vision-gps-switch.lua": lua_script_path()}

In [ ]:
ardupilot_files = write_ardupilot_files(route, inputs, package_dir / "ardupilot-src")

In [ ]:
request = PackageRequest(result, route, inputs, config, satellite, REPO_ROOT / DRONE_ONNX, CameraConfig(),
                         ardupilot_files, mission_id=MISSION_ID)

In [ ]:
archive = write_package(package_dir, request)
print(archive)

In [ ]:
package = load_package(archive)
print(package.mission_id, package.created_utc, len(package.waypoints), "out-and-back waypoints")

In [ ]:
assert newest_package(package_dir) == archive

In [ ]:
def file_table(root) -> pd.DataFrame:
    rows = [{"file": str(p.relative_to(root)), "mb": p.stat().st_size / 1e6} for p in sorted(root.rglob("*")) if p.is_file()]
    return pd.DataFrame(rows).round({"mb": 3})

In [ ]:
sizes = file_table(package.root)
sizes

In [ ]:
print(f"zip {archive.stat().st_size / 1e6:.1f} MB; folder {sizes.mb.sum():.1f} MB")

In [ ]:
def corridor_summary(package, width) -> dict:
    classes = package.reference.classes
    width_m, height_m = classes.size_m
    return {"half_width_m": width.half_width_m, "drift_m": width.drift_m,
            "footprint_half_diagonal_m": width.footprint_half_diagonal_m,
            "search_half_range_m": width.search_half_range_m,
            "raster_px": classes.pixels.shape, "raster_m": (round(width_m), round(height_m)),
            "area_px": result.class_map.pixels.shape}

In [ ]:
width = corridor_width(result.blind_limit_m, inputs, config)
corridor_summary(package, width)

In [ ]:
replay = replay_stub(package, inputs.speed_mps, inputs.altitude_m, REPLAY_PERIOD_S, STUB_NOISE_M, STUB_SEED)

In [ ]:
def replay_table(replay, route) -> pd.DataFrame:
    return pd.DataFrame([{
        "steps": len(replay.fixes), "out_and_back_m": round(replay.length_m),
        "dropped_share": round(replay.dropped_share, 3), "planner_blind_share_one_way": round(route.blind_share, 3),
        "longest_dropped_run_m": round(replay.longest_dropped_run_m()), "planner_longest_blind_m": round(route.longest_blind_m),
        "blind_limit_m": round(route.blind_limit_m), "mean_error_m": round(float(replay.error_m().mean()), 2)}])

In [ ]:
replay_table(replay, route)

In [ ]:
replay.reasons()

# Plots

In [ ]:
def package_image(package) -> GeoImage:
    classes = package.reference.classes
    return classes.with_pixels(colour_labels(classes.pixels.argmax(-1)))

In [ ]:
def draw_route(ax, package) -> None:
    waypoints = package.waypoints
    ax.plot([p.lon for p in waypoints], [p.lat for p in waypoints], color="white", lw=1.5)
    for stretch in package.manifest["route"]["blind_stretches"]:
        ax.plot([stretch["start"][1], stretch["end"][1]], [stretch["start"][0], stretch["end"][0]], color="red", lw=4)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 14), constrained_layout=True)
show_raster(axes[0], result.class_map.with_pixels(colour_labels(result.class_map.pixels.argmax(-1))))
reference = package.reference
box = [reference.classes.top_left, reference.classes.bottom_right]
axes[0].plot([box[0].lon, box[1].lon, box[1].lon, box[0].lon, box[0].lon],
             [box[0].lat, box[0].lat, box[1].lat, box[1].lat, box[0].lat], color="yellow")
axes[0].set_title("planning area class map, package corridor extent in yellow")
show_raster(axes[1], package_image(package))
draw_route(axes[1], package)
axes[1].set_title("reference/classes.tif (zero outside the strip), route white, blind stretches red")
show_raster(axes[2], reference.blind, cmap="RdYlGn_r", vmin=0, vmax=1)
draw_route(axes[2], package)
axes[2].set_title("reference/blind.tif (1 outside the strip)")
fig.savefig(figures / "package-corridor.png", dpi=110)

In [ ]:
def longest_stretch_centre(package) -> LatLon:
    stretches = package.manifest["route"]["blind_stretches"]
    longest = max(stretches, key=lambda s: s["length_m"])
    return LatLon((longest["start"][0] + longest["end"][0]) / 2, (longest["start"][1] + longest["end"][1]) / 2)

In [ ]:
def package_satellite(package) -> GeoImage:
    return GeoImage(np.asarray(Image.open(package.satellite_path)), *package.satellite_corners)

In [ ]:
centre = longest_stretch_centre(package) if package.manifest["route"]["blind_stretches"] else package.waypoints[0]
zoom = {"satellite.jpg": package_satellite(package).window(centre, ZOOM_HALF_SIZE_M),
        "classes.tif": package_image(package).window(centre, ZOOM_HALF_SIZE_M),
        "blind.tif": package.reference.blind.window(centre, ZOOM_HALF_SIZE_M)}

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
for ax, (name, raster) in zip(axes, zoom.items()):
    show_raster(ax, raster, **({"cmap": "RdYlGn_r", "vmin": 0, "vmax": 1} if name == "blind.tif" else {}))
    draw_route(ax, package)
    ax.set_xlim(raster.top_left.lon, raster.bottom_right.lon)
    ax.set_ylim(raster.bottom_right.lat, raster.top_left.lat)
    ax.set_title(name)
fig.savefig(figures / "package-zoom.png", dpi=110)

In [ ]:
def plot_replay_track(ax, replay, package) -> None:
    lats, lons = zip(*replay.positions)
    dropped = replay.dropped
    show_raster(ax, package_image(package))
    ax.scatter(np.array(lons)[~dropped], np.array(lats)[~dropped], s=3, color="lime", label="fix sent")
    ax.scatter(np.array(lons)[dropped], np.array(lats)[dropped], s=6, color="red", label="fix dropped (blind)")
    ax.legend(loc="lower left")

In [ ]:
def plot_replay_profile(ax, replay, route) -> None:
    ax.step(replay.along_m, replay.dropped.astype(float), where="post", color="red", label="dropped")
    for stretch in route.blind_stretches:
        ax.axvspan(stretch.start_m, stretch.end_m, color="red", alpha=0.15)
        ax.axvspan(replay.length_m - stretch.end_m, replay.length_m - stretch.start_m, color="red", alpha=0.15)
    ax.set_xlabel("distance flown, out and back, m")
    ax.set_ylabel("dropped (1) / sent (0)")
    ax.legend(loc="upper right")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 9), constrained_layout=True)
plot_replay_track(axes[0], replay, package)
plot_replay_profile(axes[1], replay, route)
fig.savefig(figures / "replay-dropped-fixes.png", dpi=110)